# Phase 10 — FastAPI Prediction Service

This notebook explains and exercises the API contract. Start the API in a separate PowerShell terminal from the project root:

```powershell
uvicorn api.main:app --reload --host 127.0.0.1 --port 8000
```

Then run the cells below. The service loads the model artifacts once and caches them. SHAP itself is initialized on the first request that asks for an explanation.

## 1. API architecture

`api/schemas.py` validates the request using Pydantic. `api/main.py` routes the request and handles HTTP errors. `src/evaluation/shap_explainer.py` loads XGBoost and the policy, orders features exactly as the model expects, returns a score, and optionally calculates SHAP contributions.

Endpoints:

- `GET /health`: process/artifact availability.
- `GET /model-info`: model metadata, input feature names, threshold and library versions.
- `POST /predict`: score a transaction; SHAP is included by default. Use `?include_shap=false` for score-only inference.
- `POST /explain`: deprecated compatibility alias for `/predict` with SHAP enabled.

The `Class` target column must never be sent as an input feature.

In [ ]:
import json
from pathlib import Path
import httpx

BASE_URL = "http://127.0.0.1:8000"
client = httpx.Client(base_url=BASE_URL, timeout=60.0)

health = client.get("/health")
health.status_code, health.json()

## 2. Inspect model metadata

Model metadata lets clients confirm the deployed feature contract and the current decision threshold without reading internal model files.

In [ ]:
model_info_response = client.get("/model-info")
model_info_response.raise_for_status()
model_info = model_info_response.json()
{key: model_info[key] for key in ["model_version", "model_type", "policy_version", "feature_count", "decision_threshold", "shap_available"]}

In [ ]:
model_info["feature_names"]

## 3. Build a complete request from one dataset row

For local learning only, this loads the public sample CSV. A real client should supply the feature vector from its transaction system. We drop `Class` because it is the answer used for offline evaluation, not a model input.

In [ ]:
import pandas as pd

data_path = Path("data/raw/creditcard.csv")
row = pd.read_csv(data_path, nrows=1).drop(columns=["Class"], errors="ignore").iloc[0]
payload = {key: float(value) for key, value in row.to_dict().items()}
assert len(payload) == 30
list(payload.items())[:5]

## 4. Predict and explain

The response includes the uncalibrated model score, the Phase 6 threshold decision, the risk band, latency, and the largest positive/negative SHAP contributions. SHAP explains how model features contribute relative to the selected background dataset; it does not prove real-world causation.

In [ ]:
prediction_response = client.post("/predict?include_shap=true&top_k=5", json=payload)
prediction_response.raise_for_status()
prediction = prediction_response.json()
{key: prediction[key] for key in ["request_id", "model_score", "risk_score_0_100", "decision_threshold", "risk_level", "decision", "latency_ms", "shap_available", "shap_additivity_error"]}

In [ ]:
pd.DataFrame(prediction["top_positive_contributions"])

In [ ]:
pd.DataFrame(prediction["top_negative_contributions"])

## 5. Score-only inference

When lower latency is preferred, skip SHAP calculation. This does not change the model or decision policy; it only omits the local attribution calculation.

In [ ]:
score_only_response = client.post("/predict?include_shap=false", json=payload)
score_only_response.raise_for_status()
score_only = score_only_response.json()
{key: score_only[key] for key in ["model_score", "risk_level", "decision", "shap_available", "latency_ms"]}

## 6. Error handling: malformed inputs

Pydantic returns HTTP 422 for missing required fields, extra fields (including `Class`), non-finite numbers, negative `Time`/`Amount`, and invalid `top_k`. Missing or invalid model artifacts return HTTP 503. Unexpected inference failures return a generic HTTP 500 message rather than exposing local paths or stack traces.

In [ ]:
bad_payload = dict(payload)
bad_payload.pop("V28")
bad_response = client.post("/predict", json=bad_payload)
bad_response.status_code, bad_response.json()

## 7. Response interpretation and production caveats

- `model_score` is XGBoost's current output score, not a calibrated real-world fraud probability.
- `risk_score_0_100` is exactly `100 * model_score`; it is a display transformation, not a percentage probability.
- `decision` uses the threshold stored in `models/decision_policy.json`.
- `risk_level` is an ordinal triage band defined by the same policy.
- `shap_additivity_error` should be small (near floating-point tolerance) when SHAP values reconstruct the model output correctly.
- This demo API has no authentication, authorization, rate limiting, or transport configuration for production. Do not expose it publicly until those controls are added.

## Interview questions

1. Why load the model once instead of for every request?
2. Why must the API enforce the same feature order as training?
3. Why distinguish liveness from model artifact readiness?
4. Why can SHAP add latency, and how does `include_shap=false` help?
5. Why is a model score not automatically a calibrated probability?